# Cold Data Download (ERA5) — Quickstart

## What this notebook is for
Use this notebook to **download and store the Cold hazard input datasets** required by the exposure analysis.

## When to use it
Run this notebook **once** (or whenever you want to update the dataset) **before** running the exposure notebooks.

## What you will get
At the end, the downloaded files will be saved in:
`../data/hazards/cold/`

These files will then be automatically reused by the exposure pipeline.



## Step 1 — Set the download parameters (edit if needed)

In the next cell, you can adjust:
- **YEARS / MONTHS**: time period to download
- **COUNTRIES**: area(s) to download (bounding boxes)
- **OUT_DIR**: where files will be saved (recommended: keep default)

If you are not sure, keep the default values and run the notebook as-is.


## Step 2 — Run the download

Run the notebook **from top to bottom**:
- In Jupyter: `Run > Run All`
- Or execute each cell sequentially

The download may take several minutes depending on your connection and the selected time period.


import os, gc
from pathlib import Path
import numpy as np
import xarray as xr
import rioxarray
import cdsapi
import rasterio

# CPU stability
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

# Global parameters
YEARS  = list(range(2010, 2025))
MONTHS = list(range(1, 13))

ROOT    = Path(".")
OUT_DIR = ROOT / "../data/hazards/cold"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Country bounding boxes (ERA5 format: [N, W, S, E])
COUNTRIES = {
   #  "TJK": {"area": [41.1, 67.3, 36.5, 75.2]},
    "TKM": {"area": [42.8, 52.2, 35.0, 66.8]},
   #  "KGZ": {"area": [43.3, 69.0, 39.0, 80.0]},
    # "KAZ": {"area": [55.5, 46.0, 40.0, 87.5]},
   #  "UZB": {"area": [46.0, 55.0, 37.0, 74.0]},
}

# Helpers
def cds_client():
    return cdsapi.Client()

def download_hourly_month(c, year, month, out_nc, area_bbox):
    if out_nc.exists():
        print(f"[skip] {out_nc.name}")
        return
    print(f"[download] ERA5 hourly t2m {year}-{month:02d}")
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": "2m_temperature",
            "year": str(year),
            "month": f"{month:02d}",
            "day": [f"{d:02d}" for d in range(1, 32)],
            "time": [f"{h:02d}:00" for h in range(24)],
            "area": area_bbox,
            "format": "netcdf",
        },
        str(out_nc),
    )

def detect_time_dim(da):
    if "time" in da.dims:
        return "time"
    if "valid_time" in da.dims:
        return "valid_time"
    raise ValueError("No time dimension found.")

def ensure_spatial(da):
    da = da.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)
    da = da.rio.write_crs("EPSG:4326", inplace=False)
    return da

def quick_stats(path):
    with rasterio.open(path) as src:
        arr = src.read(1)
        nod = src.nodata
    arr = arr.astype("float32")
    if nod is not None:
        vals = arr[arr != nod]
    else:
        vals = arr[np.isfinite(arr)]
    return (
        float(np.nanmin(vals)),
        float(np.nanmean(vals)),
        float(np.nanmax(vals)),
    )

# Main loop
c = cds_client()

for COUNTRY, cfg in COUNTRIES.items():
    area_bbox = cfg["area"]
    print("\n" + "#" * 80)
    print(f"### COUNTRY: {COUNTRY} | AREA = {area_bbox}")
    print("#" * 80)

    TMP_DIR = ROOT / f"./_tmp_era5_{COUNTRY.lower()}_cold_allmonths"
    TMP_DIR.mkdir(parents=True, exist_ok=True)

    OUT_TNN = OUT_DIR / f"TNn_mean_2010_2024_{COUNTRY}.tif"
    OUT_FD  = OUT_DIR / f"FD_mean_2010_2024_{COUNTRY}.tif"

    tnn_years = []
    fd_years  = []

    for y in YEARS:
        print(f"\n=== {COUNTRY} — YEAR {y} ===")
        tnn_year = None
        fd_year  = None

        for m in MONTHS:
            hourly_nc = TMP_DIR / f"era5_t2m_hourly_{COUNTRY.lower()}_{y}_{m:02d}.nc"
            download_hourly_month(c, y, m, hourly_nc, area_bbox)

            print(f"[process] {hourly_nc.name}")
            ds = xr.open_dataset(hourly_nc, chunks={"time": 240})
            da_hour = (ds["t2m"] - 273.15).rename("t2m")

            tdim = detect_time_dim(da_hour)

            daily_tmin = da_hour.resample({tdim: "1D"}).min(skipna=True)
            fd_month   = (daily_tmin < 0).sum(dim=tdim).astype("float32")
            tnn_month  = daily_tmin.min(dim=tdim)

            tnn_year = tnn_month if tnn_year is None else xr.ufuncs.minimum(tnn_year, tnn_month)
            fd_year  = fd_month if fd_year is None else (fd_year + fd_month)

            ds.close()
            del ds, da_hour, daily_tmin, fd_month, tnn_month
            gc.collect()

        tnn_year = ensure_spatial(tnn_year).astype("float32")
        fd_year  = ensure_spatial(fd_year).astype("float32")

        tnn_years.append(tnn_year)
        fd_years.append(fd_year)

        print(
            f"[{COUNTRY} {y}] "
            f"TNn range {float(tnn_year.min()):.1f}..{float(tnn_year.max()):.1f} °C | "
            f"FD range {float(fd_year.min()):.0f}..{float(fd_year.max()):.0f}"
        )

    print(f"\n[aggregate] Multi-annual means for {COUNTRY}")

    tnn_stack = xr.concat(tnn_years, dim="year")
    fd_stack  = xr.concat(fd_years,  dim="year")

    tnn_mean = tnn_stack.mean(dim="year", skipna=True)
    fd_mean  = fd_stack.mean(dim="year", skipna=True)

    tnn_mean.rio.to_raster(OUT_TNN)
    fd_mean.rio.to_raster(OUT_FD)

    print("\n✔ Written:")
    print(" -", OUT_TNN)
    print(" -", OUT_FD)
    print("TNn stats:", quick_stats(OUT_TNN))
    print("FD stats :", quick_stats(OUT_FD))


## Troubleshooting (optional)

If your kernel crashes or your machine becomes unresponsive, you can limit CPU threads by running the next cell once, then re-running the notebook.


In [1]:
# -------------------
#  In case of the kernel is full
# -------------------
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"